**bold text**#  Career Recommender Search Engine
### Built with Inverted Index + TF-IDF Ranking

**Domain:** Career Recommendation based on Hobbies & Interests  
**Dataset:** CareerRecommenderDataset.csv  
**Features:**
- Custom Inverted Index (no Elasticsearch/Whoosh)
- TF-IDF Ranking
- Boolean AND/OR queries
- Google-like Web UI via Gradio
- Keyword highlighting in results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##  Step 1: Install Dependencies

In [ ]:
!pip install gradio nltk pandas numpy -q
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
print(' All dependencies installed!')

✅ All dependencies installed!


##  Step 2: Upload Dataset


In [ ]:
from google.colab import files
import io

print('/content/drive/MyDrive/SearchEngine/CareerRecommenderDataset.csv')
uploaded = files.upload()

import pandas as pd
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f' Dataset loaded! Shape: {df.shape}')
print(f'Columns: {list(df.columns[:5])} ... (and {len(df.columns)-5} more)')
print(f'Sample career options: {df["Career_Options"].unique()[:5]}')

/content/drive/MyDrive/SearchEngine/CareerRecommenderDataset.csv


Saving CareerRecommenderDataset.csv to CareerRecommenderDataset.csv
✅ Dataset loaded! Shape: (3536, 61)
Columns: ['Drawing', 'Dancing', 'Singing', 'Sports', 'Video_Game'] ... (and 56 more)
Sample career options: ['Business Analyst, Marketing Executive, HR Manager, Operations Manager'
 'Event Manager, Wedding Planner, PR Executive'
 'Lawyer, Legal Advisor, Judge, Public Prosecutor'
 'Journalist, News Anchor, Content Writer, Editor'
 'Fashion Designer, Stylist, Fashion Illustrator']


##  Step 3: Preprocessing

In [ ]:
import re
import math
import numpy as np
from collections import defaultdict
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

# ─────────────────────────────────────────────
# Build a rich text document per unique career
# ─────────────────────────────────────────────
def build_documents(df):
    """
    Each document = one unique course/career combination.
    Text = course name + career options + active interest columns.
    Returns list of dicts with id, course, careers, interests, full_text.
    """
    interest_cols = [c for c in df.columns if c not in ['Courses', 'Career_Options']]
    documents = []
    doc_id = 0

    # Group by course to get unique career profiles
    for course, group in df.groupby('Courses'):
        career_options = group['Career_Options'].iloc[0]

        # Find which interests are commonly 'Yes' for this course
        yes_interests = []
        for col in interest_cols:
            yes_count = (group[col] == 'Yes').sum()
            if yes_count > 0:
                yes_interests.append(col.replace('_', ' '))

        # Build a rich text for indexing
        full_text = f"{course} {career_options} {' '.join(yes_interests)}"

        documents.append({
            'doc_id': doc_id,
            'course': course,
            'careers': career_options,
            'interests': yes_interests,
            'full_text': full_text
        })
        doc_id += 1

    return documents


def preprocess(text):
    """Lowercase → remove special chars → tokenize → remove stopwords → stem."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 1]
    return tokens


documents = build_documents(df)
print(f' Built {len(documents)} unique career documents')
print(f'\nSample document:')
sample = documents[0]
print(f'  Course  : {sample["course"]}')
print(f'  Careers : {sample["careers"]}')
print(f'  Interests (first 5): {sample["interests"][:5]}')

 Built 45 unique career documents

Sample document:
  Course  : Animation, Graphics and Multimedia
  Careers : Animator, VFX Artist, Multimedia Designer
  Interests (first 5): ['Drawing', 'Video Game', 'Photography', 'Coding', 'Designing']


###  Step 4: Build Inverted Index

In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)

class InvertedIndex:
    """
    Custom Inverted Index implementation.
    Structure: { term -> { doc_id -> [positions] } }
    Also stores term frequency (TF) and document frequency (DF).
    """

    def __init__(self):
        # term -> { doc_id -> [position_list] }
        self.index = defaultdict(lambda: defaultdict(list))
        # doc_id -> total token count
        self.doc_lengths = {}
        # total number of documents
        self.num_docs = 0
        # store original documents
        self.documents = []

    def build(self, documents):
        """Build inverted index from list of document dicts."""
        self.documents = documents
        self.num_docs = len(documents)

        for doc in documents:
            tokens = preprocess(doc['full_text'])
            self.doc_lengths[doc['doc_id']] = len(tokens)

            for position, term in enumerate(tokens):
                self.index[term][doc['doc_id']].append(position)

        print(f' Inverted Index built!')
        print(f'   Total unique terms : {len(self.index)}')
        print(f'   Total documents    : {self.num_docs}')

    def get_postings(self, term):
        """Return posting list (doc_ids) for a term."""
        stemmed = stemmer.stem(term.lower())
        return set(self.index.get(stemmed, {}).keys())

    def tf(self, term, doc_id):
        """Term Frequency: raw count / doc length."""
        stemmed = stemmer.stem(term.lower())
        count = len(self.index.get(stemmed, {}).get(doc_id, []))
        length = self.doc_lengths.get(doc_id, 1)
        return count / length if length > 0 else 0

    def idf(self, term):
        """Inverse Document Frequency: log(N / df + 1)."""
        stemmed = stemmer.stem(term.lower())
        df = len(self.index.get(stemmed, {}))
        return math.log((self.num_docs + 1) / (df + 1)) + 1

    def tf_idf(self, term, doc_id):
        """TF-IDF score for a term in a document."""
        return self.tf(term, doc_id) * self.idf(term)

    def boolean_and(self, terms):
        """AND query: intersection of posting lists."""
        if not terms:
            return set()
        result = self.get_postings(terms[0])
        for term in terms[1:]:
            result = result & self.get_postings(term)
        return result

    def boolean_or(self, terms):
        """OR query: union of posting lists."""
        result = set()
        for term in terms:
            result = result | self.get_postings(term)
        return result

    def stats(self):
        """Print index statistics."""
        top_terms = sorted(
            [(t, len(p)) for t, p in self.index.items()],
            key=lambda x: -x[1]
        )[:10]
        print('\nTop 10 most common index terms:')
        for term, df in top_terms:
            print(f'  {term:20s} → {df} docs')


# Build the index
inv_index = InvertedIndex()
inv_index.build(documents)
inv_index.stats()

 Inverted Index built!
   Total unique terms : 238
   Total documents    : 45

Top 10 most common index terms:
  research             → 21 docs
  mathemat             → 16 docs
  physic               → 15 docs
  scienc               → 15 docs
  english              → 14 docs
  code                 → 12 docs
  bachelor             → 12 docs
  solv                 → 12 docs
  puzzl                → 12 docs
  engeeni              → 12 docs


##  Step 5: Query Engine (TF-IDF Ranking + Boolean)

In [ ]:
class QueryEngine:
    """
    Handles query parsing, boolean retrieval, and TF-IDF ranking.
    Supports:
      - Single word queries
      - Multi-word queries (default: OR with TF-IDF ranking)
      - Explicit AND / OR boolean operators
    """

    def __init__(self, inv_index):
        self.idx = inv_index

    def parse_query(self, raw_query):
        """
        Parse query string.
        Returns (mode, terms) where mode is 'AND' or 'OR'.
        """
        raw = raw_query.strip()

        # Detect explicit boolean operators
        if ' AND ' in raw.upper():
            parts = re.split(r'\bAND\b', raw, flags=re.IGNORECASE)
            terms = [p.strip() for p in parts if p.strip()]
            return 'AND', terms
        elif ' OR ' in raw.upper():
            parts = re.split(r'\bOR\b', raw, flags=re.IGNORECASE)
            terms = [p.strip() for p in parts if p.strip()]
            return 'OR', terms
        else:
            # Multi-word: treat as OR with TF-IDF ranking
            terms = raw.split()
            return 'OR', terms

    def score_document(self, doc_id, terms):
        """Calculate aggregate TF-IDF score for a document across all query terms."""
        score = 0.0
        for term in terms:
            score += self.idx.tf_idf(term, doc_id)
        return score

    def search(self, raw_query, top_k=10):
        """
        Main search function.
        Returns list of ranked result dicts with score, doc info.
        """
        if not raw_query.strip():
            return []

        mode, terms = self.parse_query(raw_query)

        # Boolean retrieval
        if mode == 'AND':
            candidate_ids = self.idx.boolean_and(terms)
        else:
            candidate_ids = self.idx.boolean_or(terms)

        if not candidate_ids:
            return []

        # TF-IDF ranking
        scored = []
        for doc_id in candidate_ids:
            score = self.score_document(doc_id, terms)
            doc = self.idx.documents[doc_id]
            scored.append({
                'doc_id': doc_id,
                'score': round(score, 4),
                'course': doc['course'],
                'careers': doc['careers'],
                'interests': doc['interests'],
                'mode': mode,
                'query_terms': terms
            })

        # Sort by score descending
        scored.sort(key=lambda x: -x['score'])
        return scored[:top_k]


engine = QueryEngine(inv_index)

# Quick test
test_results = engine.search('coding mathematics')
print('Test query: "coding mathematics"')
for r in test_results[:3]:
    print(f'  [{r["score"]}] {r["course"]} → {r["careers"][:60]}...')

Test query: "coding mathematics"
  [0.3628] B.Sc. Mathematics → Statistician, Data Scientist, Actuary...
  [0.3042] BCA- Bachelor of Computer Applications → Software Developer, System Analyst, Web Developer...
  [0.2242] B.Tech in Machine Learning → ML Engineer, AI Researcher, Data Scientist...


##  Step 7: Test Queries in Console (Optional)

In [ ]:
# ── Test various query types ──────────────────

test_queries = [
    'coding',                        # Single word
    'coding mathematics',            # Multi-word OR
    'music AND dancing',             # Explicit AND
    'biology OR chemistry',          # Explicit OR
    'travelling photography',        # Multi-word
    'accounting economics business', # 3-word OR
]

for q in test_queries:
    results = engine.search(q, top_k=3)
    print(f"\n Query: '{q}'  →  {len(results)} results")
    print(f"   Mode: {results[0]['mode'] if results else 'N/A'}")
    for r in results:
        print(f"   [{r['score']:.4f}] {r['course'][:50]:50s} → {r['careers'][:55]}")


 Query: 'coding'  →  3 results
   Mode: OR
   [0.1617] BVA- Bachelor of Visual Arts                       → Visual Artist, Illustrator, Art Director
   [0.1617] BCA- Bachelor of Computer Applications             → Software Developer, System Analyst, Web Developer
   [0.1415] Animation, Graphics and Multimedia                 → Animator, VFX Artist, Multimedia Designer

 Query: 'coding mathematics'  →  3 results
   Mode: OR
   [0.3628] B.Sc. Mathematics                                  → Statistician, Data Scientist, Actuary
   [0.3042] BCA- Bachelor of Computer Applications             → Software Developer, System Analyst, Web Developer
   [0.2242] B.Tech in Machine Learning                         → ML Engineer, AI Researcher, Data Scientist

 Query: 'music AND dancing'  →  0 results
   Mode: N/A

 Query: 'biology OR chemistry'  →  3 results
   Mode: OR
   [0.6217] B.Sc. Chemistry                                    → Chemist, Lab Analyst, Chemical Quality Analyst
   [0.4544] BPharma-

##  Step 8: Index Statistics

In [ ]:
import pandas as pd

# ── Show index overview ──
print('=' * 55)
print('         INVERTED INDEX STATISTICS')
print('=' * 55)
print(f'  Total documents indexed  : {inv_index.num_docs}')
print(f'  Total unique terms       : {len(inv_index.index)}')
print(f'  Avg tokens per document  : {sum(inv_index.doc_lengths.values()) / inv_index.num_docs:.1f}')

# ── Document length distribution ──
import numpy as np
lengths = list(inv_index.doc_lengths.values())
print(f'  Min doc length           : {min(lengths)}')
print(f'  Max doc length           : {max(lengths)}')
print(f'  Median doc length        : {int(np.median(lengths))}')

# ── Top terms by document frequency ──
print('\nTop 15 Terms by Document Frequency (DF):')
top = sorted(inv_index.index.items(), key=lambda x: -len(x[1]))[:15]
df_stats = pd.DataFrame([
    {'Term': t, 'Doc Freq (DF)': len(postings), 'IDF': round(inv_index.idf(t), 4)}
    for t, postings in top
])
print(df_stats.to_string(index=False))

# ── Unique career courses ──
print(f'\n Total Unique Career Courses: {len(documents)}')
for i, doc in enumerate(documents[:10]):
    print(f'  {i+1:2d}. {doc["course"]}')
if len(documents) > 10:
    print(f'  ... and {len(documents)-10} more courses')

         INVERTED INDEX STATISTICS
  Total documents indexed  : 45
  Total unique terms       : 238
  Avg tokens per document  : 16.5
  Min doc length           : 10
  Max doc length           : 28
  Median doc length        : 15

Top 15 Terms by Document Frequency (DF):
    Term  Doc Freq (DF)    IDF
research             21 1.7376
mathemat             16 1.9954
  physic             15 2.0561
  scienc             15 2.0561
 english             14 2.1206
    code             12 2.2637
bachelor             12 2.2637
    solv             12 2.2637
   puzzl             12 2.2637
 engeeni             12 2.2637
   engin             12 2.2637
 analyst             11 2.3437
  comput             11 2.3437
    part             11 2.3437
    tech             11 2.3437

📚 Total Unique Career Courses: 45
   1. Animation, Graphics and Multimedia
   2. B.Arch- Bachelor of Architecture
   3. B.Com- Bachelor of Commerce
   4. B.Ed.
   5. B.Sc Bioinformatics
   6. B.Sc Genetics
   7. B.Sc Healthcare IT
